# 76. Minimum Window Substring
**Difficulty:** 🔴 Hard · **Topic:** String · **LeetCode:** https://leetcode.com/problems/minimum-window-substring/

## 💡 Concepts

**Core concept(s):** A **sliding window** with letter **counts** and a running "how many still needed" tally.

**Why it applies here:** We want the shortest stretch of `s` that contains all letters of `t` (with repeats). Grow the window on the right until it covers everything needed, then shrink from the left as much as possible while it still covers — recording the smallest valid window seen.

**Key intuition:** Expand until you have everything; then squeeze from the left to make it as small as possible.

---

### 📚 What is a Sliding Window?
A **window** is a range `[left, right]` over the string that you grow on the right and shrink on the left, keeping some running summary (a count, a set) as it moves. You never re-scan from scratch.
- **Complexity:** each character enters and leaves the window at most once → **O(n)** total.
- **In Python:** two indices plus a `dict`/`set`/`Counter` describing what's inside.

### 📚 What is a Hash Map / Hash Set?
A **hash map** (Python `dict`) stores **key → value** pairs; a **hash set** (`set`) stores unique keys. Both use a *hash function* to jump straight to a slot instead of scanning.
- **Operations & complexity:** insert / lookup / delete are **O(1) on average**.
- **In Python:** `dict` for counts/mappings, `set` for "have I seen this?". `collections.Counter` counts items for you.

---

**Prerequisite knowledge:**
- Sliding window (grow right, shrink left).
- A `Counter` of needed letters and a `missing` tally.

## 📝 Problem

Return the shortest substring of `s` that contains every character of `t` (including duplicates). Return `""` if none exists.

**Example**
```
s = "ADOBECODEBANC", t = "ABC" -> "BANC"
s = "a", t = "aa"               -> ""
```

> Two approaches: brute `O(n^2 · m)` and sliding window `O(n)`.

### Approach 1 — Check Every Start (worst)

**Idea:** From each start, extend until the window covers `t`; record the shortest.

**Time complexity:** `O(n^2 · m)` (checking coverage costs work too).

**Space complexity:** `O(m)`.

In [ ]:
from collections import Counter

def min_window_brute(s: str, t: str) -> str:
    if not t or not s:
        return ""
    need = Counter(t)                      # how many of each character we must cover
    n = len(s)
    best = ""
    for i in range(n):                     # try every starting index
        count = {}                         # characters collected in the current window
        for j in range(i, n):              # extend the window to the right
            count[s[j]] = count.get(s[j], 0) + 1
            # Does the window now contain enough of every needed character?
            if all(count.get(c, 0) >= need[c] for c in need):
                if best == "" or (j - i + 1) < len(best):
                    best = s[i:j+1]        # keep the shortest covering window
                break                      # can't get shorter for this start -> next i
    return best

### Approach 2 — Sliding Window (optimal)

**Idea:** Track how many characters are still `missing`. Grow right; when nothing is missing, shrink left to the smallest valid window, recording the best.

**Time complexity:** `O(n)`.

**Space complexity:** `O(m)`.

In [ ]:
from collections import Counter

def min_window_slide(s: str, t: str) -> str:
    if not t or not s:
        return ""
    need = Counter(t)                      # remaining count needed for each character
    missing = len(t)                       # total characters still to cover (with repeats)
    left = 0                               # left edge of the window
    best = (float("inf"), 0, 0)            # (window length, start, end+1) of the best so far
    for right, c in enumerate(s):          # right edge sweeps across s
        if need[c] > 0:                    # c is a character we still needed
            missing -= 1                   # one fewer to cover
        need[c] -= 1                       # (can go negative for extra copies)
        while missing == 0:                # window covers all of t -> try to shrink it
            if right - left + 1 < best[0]:
                best = (right - left + 1, left, right + 1)   # record a smaller window
            need[s[left]] += 1             # about to drop the leftmost char
            if need[s[left]] > 0:          # dropping it breaks coverage
                missing += 1               # we now need that character again
            left += 1                      # move the left edge right
    return "" if best[0] == float("inf") else s[best[1]:best[2]]

In [ ]:
# Correctness check
tests = [
    ("ADOBECODEBANC","ABC","BANC"),
    ("a","a","a"),
    ("a","aa",""),
    ("aa","aa","aa"),
]
for s, t, exp in tests:
    a, b = min_window_brute(s, t), min_window_slide(s, t)
    print(f"s={s!r}, t={t!r} -> brute={a!r}, slide={b!r} | expected={exp!r}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit). Sub-millisecond rows are noisy — look at the trend.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    # target never fully appears until the very end -> both do maximal work
    s = ("abc" * (n // 3 + 1))[:n - 1] + "z"
    t = "abcz"
    return (s, t)

solutions = {
    "brute O(n^2*m)": min_window_brute,
    "slide O(n)    ": min_window_slide,
}
sizes = [500, 1000, 2000, 4000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Grow-then-shrink window:** for "shortest stretch that contains X", expand until valid, then contract to minimal — the classic variable-size window.
- **A single "missing" counter:** track coverage in O(1) per step instead of re-scanning counts.
- **Signal:** "smallest/shortest substring containing all of ...", "minimum window".
- **Related problems:** Longest Substring Without Repeating, Longest Repeating Character Replacement, Permutation in String.
- **Common pitfalls:** (1) recomputing coverage each step (slow); (2) shrinking before the window is valid; (3) not restoring counts when shrinking.